In [1]:
!pip install -q --upgrade wandb

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm
import wandb
import math
import random

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_api")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb_api")

# Fix all random seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 72.8 MB/s eta 0:00:00:00:0100:01
Device: cuda


In [4]:
config = {
    # Paths
    "data_dir": Path("/kaggle/input/competitions/jaguar-re-id"),
    "checkpoint_dir": Path("checkpoints"),
    
    # Model
    "megadescriptor_model": "hf-hub:BVRA/MegaDescriptor-L-384",
    "input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    
    # Training — FIXED across all loss functions
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 25,  # reduced from 50 to save GPU time
    "patience": 7,
    "val_split": 0.2,
    "seed": SEED,
    
    # ArcFace defaults (used as reference)
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,
}

config["checkpoint_dir"].mkdir(exist_ok=True)
print("Config ready")

Config ready


In [5]:
# Load training data
train_df = pd.read_csv(config["data_dir"] / "train.csv")

print(f"Total images: {len(train_df)}")
print(f"Unique identities: {train_df['ground_truth'].nunique()}")

# Encode labels
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['ground_truth'])
num_classes = len(label_encoder.classes_)

# Stratified split — same seed every time, this never changes
train_data, val_data = train_test_split(
    train_df,
    test_size=config["val_split"],
    random_state=config["seed"],
    stratify=train_df['ground_truth']
)

print(f"Train: {len(train_data)} images")
print(f"Val: {len(val_data)} images")
print(f"Num classes: {num_classes}")

Total images: 1895
Unique identities: 31
Train: 1516 images
Val: 379 images
Num classes: 31


In [6]:
# Load MegaDescriptor
print("Loading MegaDescriptor...")
megadescriptor = timm.create_model(
    config["megadescriptor_model"],
    pretrained=True
)
megadescriptor.eval()
megadescriptor.to(device)

# Get embedding dimension
with torch.no_grad():
    dummy = torch.randn(1, 3, config["input_size"], config["input_size"]).to(device)
    megadescriptor_dim = megadescriptor(dummy).shape[1]

print(f"MegaDescriptor embedding dim: {megadescriptor_dim}")

# Preprocessing
preprocess = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_embeddings(model, image_paths, batch_size=32):
    model.eval()
    embeddings = []
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        tensors = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                tensors.append(preprocess(img))
            except:
                tensors.append(torch.zeros(3, config["input_size"], config["input_size"]))
        batch = torch.stack(tensors).to(device)
        embeddings.append(model(batch).cpu().numpy())
    return np.vstack(embeddings)

# Extract and cache
cache_dir = Path("embeddings")
cache_dir.mkdir(exist_ok=True)

train_paths = [config["data_dir"] / "train/train" / f for f in train_data["filename"].values]
val_paths = [config["data_dir"] / "train/train" / f for f in val_data["filename"].values]

train_cache = cache_dir / "train_embeddings.npz"
val_cache = cache_dir / "val_embeddings.npz"

if train_cache.exists():
    train_embeddings = np.load(train_cache)["embeddings"]
    print(f"Loaded cached train embeddings: {train_embeddings.shape}")
else:
    print("Extracting train embeddings...")
    train_embeddings = extract_embeddings(megadescriptor, train_paths)
    np.savez_compressed(train_cache, embeddings=train_embeddings)
    print(f"Saved train embeddings: {train_embeddings.shape}")

if val_cache.exists():
    val_embeddings = np.load(val_cache)["embeddings"]
    print(f"Loaded cached val embeddings: {val_embeddings.shape}")
else:
    print("Extracting val embeddings...")
    val_embeddings = extract_embeddings(megadescriptor, val_paths)
    np.savez_compressed(val_cache, embeddings=val_embeddings)
    print(f"Saved val embeddings: {val_embeddings.shape}")

Loading MegaDescriptor...


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

MegaDescriptor embedding dim: 1536
Extracting train embeddings...


  0%|          | 0/48 [00:00<?, ?it/s]

Saved train embeddings: (1516, 1536)
Extracting val embeddings...


  0%|          | 0/12 [00:00<?, ?it/s]

Saved val embeddings: (379, 1536)


In [7]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


def compute_validation_map(model, val_embeddings, val_labels):
    model.eval()
    with torch.no_grad():
        val_tensor = torch.FloatTensor(val_embeddings).to(device)
        finetuned_emb = model.get_embeddings(val_tensor).cpu().numpy()
    
    sim_matrix = cosine_similarity(finetuned_emb)
    np.fill_diagonal(sim_matrix, -1)
    
    identity_aps = {}
    for query_idx in range(len(val_labels)):
        query_label = val_labels[query_idx]
        similarities = sim_matrix[query_idx]
        is_match = (val_labels == query_label).astype(int)
        is_match[query_idx] = 0
        
        sorted_indices = np.argsort(-similarities)
        sorted_matches = is_match[sorted_indices]
        
        n_positives = sorted_matches.sum()
        if n_positives == 0:
            continue
        
        cumsum = np.cumsum(sorted_matches)
        precision_at_k = cumsum / np.arange(1, len(sorted_matches) + 1)
        ap = np.sum(precision_at_k * sorted_matches) / n_positives
        
        if query_label not in identity_aps:
            identity_aps[query_label] = []
        identity_aps[query_label].append(ap)
    
    identity_mean_aps = [np.mean(aps) for aps in identity_aps.values()]
    return np.mean(identity_mean_aps)

print("Dataset and validation function ready")

Dataset and validation function ready


In [8]:
# ── 1. ArcFace (baseline loss) ──────────────────────────────────────────
class ArcFaceLayer(nn.Module):
    def __init__(self, embedding_dim, num_classes, margin=0.5, scale=64.0):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, weight).clamp(-1+1e-6, 1-1e-6)
        theta = torch.acos(cosine)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.scale * torch.cos(theta + one_hot * self.margin)
        return F.cross_entropy(output, labels)


# ── 2. CosFace ───────────────────────────────────────────────────────────
class CosFaceLayer(nn.Module):
    def __init__(self, embedding_dim, num_classes, margin=0.35, scale=64.0):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, weight)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.scale * (cosine - one_hot * self.margin)
        return F.cross_entropy(output, labels)


# ── 3. Triplet Loss (hard mining) ────────────────────────────────────────
class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        dist = torch.cdist(embeddings, embeddings, p=2)
        pos_mask = labels.unsqueeze(0) == labels.unsqueeze(1)
        neg_mask = ~pos_mask
        pos_mask.fill_diagonal_(False)

        hardest_pos = (dist * pos_mask.float()).max(dim=1)[0]
        hardest_neg = (dist + 1e6 * (~neg_mask).float()).min(dim=1)[0]

        loss = F.relu(hardest_pos - hardest_neg + self.margin)
        return loss.mean()


# ── 4. ArcFace + Triplet Combined ────────────────────────────────────────
class CombinedLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, alpha=0.5):
        super().__init__()
        self.arcface = ArcFaceLayer(embedding_dim, num_classes)
        self.triplet = TripletLoss()
        self.alpha = alpha

    def forward(self, embeddings, labels):
        return (self.alpha * self.arcface(embeddings, labels) +
                (1 - self.alpha) * self.triplet(embeddings, labels))


# ── 5. Focal Loss ────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, gamma=2.0, scale=64.0):
        super().__init__()
        self.gamma = gamma
        self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        logits = self.scale * F.linear(embeddings, weight)
        ce = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma * ce).mean()
        return loss


# ── 6. SubCenter ArcFace ─────────────────────────────────────────────────
class SubCenterArcFace(nn.Module):
    def __init__(self, embedding_dim, num_classes, K=3, margin=0.5, scale=64.0):
        super().__init__()
        self.K = K
        self.num_classes = num_classes
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(
            torch.FloatTensor(num_classes * K, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, weight)
        cosine = cosine.view(-1, self.num_classes, self.K).max(dim=2)[0]
        cosine = cosine.clamp(-1+1e-6, 1-1e-6)
        theta = torch.acos(cosine)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.scale * torch.cos(theta + one_hot * self.margin)
        return F.cross_entropy(output, labels)

print("All 6 loss functions defined")

All 6 loss functions defined


In [9]:
class EmbeddingProjection(nn.Module):
    def __init__(self, input_dim=1536, hidden_dim=512, 
                 output_dim=256, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim),
        )
    
    def forward(self, x):
        return self.network(x)
    
    def get_embeddings(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)

print("Embedding model defined")

Embedding model defined


In [10]:
def train_one_epoch(model, loss_fn, loader, optimizer, device):
    model.train()
    loss_fn.train()
    total_loss = 0

    for embeddings, labels in tqdm(loader, desc="Training", leave=False):
        embeddings, labels = embeddings.to(device), labels.to(device)

        optimizer.zero_grad()
        projected = model(embeddings)
        loss = loss_fn(projected, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate_one_epoch(model, loss_fn, loader, device):
    model.eval()
    loss_fn.eval()
    total_loss = 0

    with torch.no_grad():
        for embeddings, labels in tqdm(loader, desc="Validation", leave=False):
            embeddings, labels = embeddings.to(device), labels.to(device)
            projected = model(embeddings)
            loss = loss_fn(projected, labels)
            total_loss += loss.item()

    return total_loss / len(loader)


print("Training functions defined")

Training functions defined


In [11]:
# Define all 6 experiments
experiments = [
    {
        "name": "loss-arcface",
        "loss_fn": lambda: ArcFaceLayer(config["embedding_dim"], num_classes)
    },
    {
        "name": "loss-cosface",
        "loss_fn": lambda: CosFaceLayer(config["embedding_dim"], num_classes)
    },
    {
        "name": "loss-triplet",
        "loss_fn": lambda: TripletLoss()
    },
    {
        "name": "loss-combined-arcface-triplet",
        "loss_fn": lambda: CombinedLoss(config["embedding_dim"], num_classes)
    },
    {
        "name": "loss-focal",
        "loss_fn": lambda: FocalLoss(config["embedding_dim"], num_classes)
    },
    {
        "name": "loss-subcenter-arcface",
        "loss_fn": lambda: SubCenterArcFace(config["embedding_dim"], num_classes)
    },
]

# Fixed datasets — same for all experiments
train_dataset = EmbeddingDataset(train_embeddings, train_data['label_encoded'].values)
val_dataset = EmbeddingDataset(val_embeddings, val_data['label_encoded'].values)

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], 
                          shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], 
                        shuffle=False, num_workers=0)

val_labels = val_data['ground_truth'].values

# Results table
results = []

# ── Main Loop ────────────────────────────────────────────────────────────
for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Running: {exp['name']}")
    print(f"{'='*60}")

    # Reset seeds before each experiment for fairness
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    # Initialize model and loss fresh for each experiment
    model = EmbeddingProjection(
        input_dim=megadescriptor_dim,
        hidden_dim=config["hidden_dim"],
        output_dim=config["embedding_dim"],
        dropout=config["dropout"]
    ).to(device)

    loss_fn = exp["loss_fn"]().to(device)

    # Combine model and loss parameters for optimizer
    all_params = list(model.parameters()) + list(loss_fn.parameters())
    optimizer = torch.optim.AdamW(
        all_params,
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # Initialize W&B run
    wandb.login(key=os.environ["WANDB_API_KEY"])
    run = wandb.init(
        project="jaguar-reid-mishank",
        name=exp["name"],
        config={
            **config,
            "loss_function": exp["name"],
            "num_parameters": sum(p.numel() for p in model.parameters()),
            "num_classes": num_classes,
        }
    )

    # Training loop
    best_val_loss = float('inf')
    best_val_map = 0.0
    patience_counter = 0

    for epoch in range(config["num_epochs"]):
        train_loss = train_one_epoch(model, loss_fn, train_loader, optimizer, device)
        val_loss = validate_one_epoch(model, loss_fn, val_loader, device)
        val_map = compute_validation_map(model, val_embeddings, val_labels)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        # Log to W&B
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_map": val_map,
            "learning_rate": current_lr,
        })

        print(f"Epoch {epoch+1:2d} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val mAP: {val_map:.4f}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_map = val_map
            patience_counter = 0
            checkpoint_path = config["checkpoint_dir"] / f"{exp['name']}_best.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_map": best_val_map,
                "val_loss": best_val_loss,
            }, checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= config["patience"]:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Log best results and save artifact
    wandb.log({
        "best_val_map": best_val_map,
        "best_val_loss": best_val_loss,
    })

    artifact = wandb.Artifact(f"model-{exp['name']}", type="model")
    artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(artifact)

    wandb.finish()

    # Store results
    results.append({
        "loss_function": exp["name"],
        "best_val_map": best_val_map,
        "best_val_loss": best_val_loss,
    })

    print(f"Finished {exp['name']} | Best Val mAP: {best_val_map:.4f}")

print("\n" + "="*60)
print("ALL EXPERIMENTS COMPLETE")
print("="*60)


Running: loss-arcface


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jain5 (jain5-university-of-potsdam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training:   0%|          | 0/48 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch  1 | Train Loss: 31.1585 | Val Loss: 21.8919 | Val mAP: 0.4164


...

Training:   0%|          | 0/48 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 25 | Train Loss: 0.4256 | Val Loss: 4.7063 | Val mAP: 0.6110


best_val_loss,▁
best_val_map,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
learning_rate,████████████████████████▁
train_loss,█▆▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map,▁▃▄▅▅▆▆▇▇▇▇▇▇▇███████████
best_val_loss,4.41283
best_val_map,0.61006
epoch,25
learning_rate,5e-05


Finished loss-subcenter-arcface | Best Val mAP: 0.6101

ALL EXPERIMENTS COMPLETE


In [12]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("best_val_map", ascending=False)
results_df["best_val_map"] = results_df["best_val_map"].round(4)
results_df["best_val_loss"] = results_df["best_val_loss"].round(4)

print("Loss Function Comparison Results")
print("="*50)
print(results_df.to_string(index=False))
print("="*50)
print(f"Best loss function: {results_df.iloc[0]['loss_function']}")
print(f"Best val mAP: {results_df.iloc[0]['best_val_map']}")

Loss Function Comparison Results
                loss_function  best_val_map  best_val_loss
loss-combined-arcface-triplet        0.7900         2.2577
                 loss-arcface        0.7850         4.4457
                 loss-cosface        0.7823         3.2073
                 loss-triplet        0.6837         0.0756
       loss-subcenter-arcface        0.6101         4.4128
                   loss-focal        0.4177         0.5060
Best loss function: loss-combined-arcface-triplet
Best val mAP: 0.79


In [14]:
# Load best model (combined loss)
best_model = EmbeddingProjection(
    input_dim=megadescriptor_dim,
    hidden_dim=config["hidden_dim"],
    output_dim=config["embedding_dim"],
    dropout=config["dropout"]
).to(device)

checkpoint = torch.load(
    config["checkpoint_dir"] / "loss-combined-arcface-triplet_best.pth",
    map_location=device,
    weights_only=False
)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()
print(f"Loaded best model | Val mAP: {checkpoint['val_map']:.4f}")

# Load test pairs
test_pairs_df = pd.read_csv(config["data_dir"] / "test.csv")
test_images = sorted(list(
    set(test_pairs_df['query_image'].unique()) | 
    set(test_pairs_df['gallery_image'].unique())
))

# Extract test embeddings
test_paths = [config["data_dir"] / "test/test" / f for f in test_images]
print(f"Extracting embeddings for {len(test_images)} test images...")
test_mega_embeddings = extract_embeddings(megadescriptor, test_paths)

with torch.no_grad():
    test_tensor = torch.FloatTensor(test_mega_embeddings).to(device)
    test_embeddings_proj = best_model.get_embeddings(test_tensor).cpu().numpy()

# Compute similarities
img_to_emb = {fn: emb for fn, emb in zip(test_images, test_embeddings_proj)}

similarities = []
for _, row in tqdm(test_pairs_df.iterrows(), total=len(test_pairs_df)):
    q = img_to_emb[row['query_image']]
    g = img_to_emb[row['gallery_image']]
    similarities.append(float(np.dot(q, g)))

similarities = np.clip(similarities, 0.0, 1.0)

submission_df = pd.DataFrame({
    'row_id': test_pairs_df['row_id'],
    'similarity': similarities
})

submission_df.to_csv("submission_combined_loss.csv", index=False)
print(f"Submission saved: {len(submission_df)} rows")

from IPython.display import FileLink
FileLink('submission_combined_loss.csv')

Loaded best model | Val mAP: 0.7900
Extracting embeddings for 371 test images...


  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/137270 [00:00<?, ?it/s]

Submission saved: 137270 rows


/kaggle/working/submission_combined_loss.csv